In [ ]:
import pandas as pd
import numpy as np
import itertools
import joblib

In [ ]:
#CARICAMENTO MODELLI
modello_apo = joblib.load('modello_apogeo_v2.pkl')
modello_ang = joblib.load('modello_angolo_v2.pkl')
scaler_fisso = joblib.load('scaler_v2.pkl')

In [ ]:
#RIPORTO LA FUNZIONE DI CALCOLO SETUP OTTIMALE DAL NOTEBOOK optGemini.ipynb
def calcola_setup(vento_reale, std_reale, modello_apo, modello_ang, scaler):
    lista_zavorre = [0, 20, 40, 60]
    delay_possibili = np.arange(1.5, 4.51, 0.05) 
    
    if vento_reale < 0.2:
        rampe_possibili = [0.0]
    else:
        angolo_min = vento_reale * -4.5
        angolo_max = vento_reale * -2.0
        rampe_possibili = np.arange(angolo_min, angolo_max + 0.1, 0.1)

    combinazioni = list(itertools.product([vento_reale], [std_reale], lista_zavorre, delay_possibili, rampe_possibili))
    df = pd.DataFrame(combinazioni, columns=['vento_medio', 'std_vento', 'zavorra', 'delay_secondi', 'angolo_rampa'])
    
    dati_scalati = scaler.transform(df)
    df['Prev_Apogeo'] = modello_apo.predict(dati_scalati)
    df['Prev_Errore_Inclinazione'] = modello_ang.predict(dati_scalati)
    
    tolleranza = 20.0
    df_sicuri = df[np.abs(df['Prev_Errore_Inclinazione']) <= tolleranza].copy()
    
    if df_sicuri.empty:
        return "Nessun setup sicuro trovato"
    
    ottimali = []
    for z in lista_zavorre:
        df_z = df_sicuri[df_sicuri['zavorra'] == z]
        if not df_z.empty:
            migliore = df_z.iloc[np.abs(df_z['Prev_Errore_Inclinazione']).argsort()[:1]].iloc[0]
            ottimali.append(migliore)
            
    df_finale = pd.DataFrame(ottimali)
    df_finale = df_finale.round({'delay_secondi': 2, 'angolo_rampa': 1, 'Prev_Apogeo': 1, 'Prev_Errore_Inclinazione': 1})
    
    return df_finale[['zavorra', 'angolo_rampa', 'delay_secondi', 'Prev_Apogeo', 'Prev_Errore_Inclinazione']]

In [ ]:
#GENERAZIONE SCENARI PER SIMULAZIONI X RISCHIO
venti = [1.5, 3.0, 4.5, 6.0]
turbolenze_rel = [0.05, 0.15]
zavorre = [0, 20, 40, 60]

scenari_totali = []

#CALCOLO SETUP OTTIMO PER OGNI COMBINAZIONE DI VENTO, TURBOLENZA E ZAVORRA
for v, t_rel, z in itertools.product(venti, turbolenze_rel, zavorre):
    std = round(v * t_rel, 2)
    
    tabella_ottimi = calcola_setup(v, std, modello_apo, modello_ang, scaler_fisso)
    setup_migliore = tabella_ottimi[tabella_ottimi['zavorra'] == z]
    
    if setup_migliore.empty:
        continue
        
    
    #CREAZIONE DI 10 REPLICHE PER OGNI SCENARIO
    for i in range(1, 11):
        scenari_totali.append({
            'ID_SCENARIO': f"V{v}_T{t_rel}_Z{z}",
            'vento_medio': v,
            'std_vento': std,
            'zavorra': z,
            'angolo_rampa': setup_migliore['angolo_rampa'],
            'delay_secondi': setup_migliore['delay_secondi'],
            'SIM_N': i,
            'ANGOLO_ACC_C6': '',
            'INCLINAZIONE': ''})

df_rischio_sim = pd.DataFrame(scenari_totali)
df_rischio_sim.to_csv("Simulazioni_x_Analisi_Rischio.csv", index=False)

In [ ]:
#ANALISI DEL RISCHIO
df_risultati = pd.read_csv("Simulazioni_x_Analisi_Rischio.csv")

df_risultati['ERRORE_REALE'] = (90.0 - df_risultati['ANGOLO_ACC_C6']) * np.sign(df_risultati['INCLINAZIONE'])

#AGGREGAZIONE PER SCENARIO
analisi_rischio = df_risultati.groupby(['ID_SCENARIO', 'vento_medio', 'std_vento', 'zavorra']).agg(
    Errore_Medio=('ERRORE_REALE', 'mean'),
    Dispersione_Rischio=('ERRORE_REALE', 'std'),
    Peggior_Caso_Neg=('ERRORE_REALE', 'min'),
    Peggior_Caso_Pos=('ERRORE_REALE', 'max')).reset_index()
analisi_rischio['Range_Totale'] = analisi_rischio['Peggior_Caso_Pos'] - analisi_rischio['Peggior_Caso_Neg']